In [ ]:
import kaggle_benchmarks as kbench
import random, re
import numpy as np
import pandas as pd

In [ ]:
def _parse(response, a, b):
    text = response.strip().lower()
    for token, opt in [(a.lower(), a), (b.lower(), b)]:
        if re.search(r'\b' + re.escape(token) + r'\b', text):
            return opt
    pos_a, pos_b = text.find(a.lower()), text.find(b.lower())
    if pos_a >= 0 and pos_b >= 0: return a if pos_a < pos_b else b
    if pos_a >= 0: return a
    if pos_b >= 0: return b
    return None

def _acc(choices, targets):
    correct = sum(c == t for c, t in zip(choices, targets) if c is not None)
    valid   = sum(c is not None for c in choices)
    return correct / valid if valid else 0.0

def _slope(x, y):
    if len(x) < 2: return 0.0
    x, y = np.array(x, float), np.array(y, float)
    xm, ym = x.mean(), y.mean()
    denom = ((x - xm) ** 2).sum()
    return float(((x - xm) * (y - ym)).sum() / denom) if denom else 0.0

N_ITEMS = 5

@kbench.task(
    name="Transitive Inference - SDE and Terminal Item Effect",
    description=(
        "5-item linear hierarchy. All adjacent ranking rules given in one prompt. "
        "Tests symbolic distance effect (accuracy by rank distance) and terminal "
        "item effect (endpoint pairs easier than interior pairs)."
    ),
)
def ti_sde_and_tie(llm) -> None:
    pool = [f"Item-{chr(65+i)}" for i in range(N_ITEMS)]
    random.shuffle(pool)
    labels    = pool
    endpoints = {labels[0], labels[N_ITEMS - 1]}

    adj_lines = "\n".join(
        f"  {labels[i]} ranks higher than {labels[i+1]}"
        for i in range(N_ITEMS - 1)
    )
    llm.prompt(
        f"TRANSITIVE INFERENCE TASK\n"
        f"Complete ranking rules (higher = better):\n"
        f"{adj_lines}\n\n"
        f"These rules are transitive. Reply with ONLY the item name that ranks higher. No other text."
    )

    test_pairs = [(i, j, j-i) for i in range(N_ITEMS) for j in range(i+2, N_ITEMS)]
    dist_ch: dict[int, list] = {}
    dist_tg: dict[int, list] = {}
    terminal_ch, terminal_tg = [], []
    interior_ch,  interior_tg  = [], []

    for hi, lo, dist in test_pairs:
        a, b   = labels[hi], labels[lo]
        order  = [a, b]; random.shuffle(order)
        resp   = llm.prompt(f"Which ranks higher: {order[0]} or {order[1]}?")
        choice = _parse(resp, order[0], order[1])

        dist_ch.setdefault(dist, []).append(choice)
        dist_tg.setdefault(dist, []).append(a)

        if (a in endpoints) or (b in endpoints):
            terminal_ch.append(choice); terminal_tg.append(a)
        else:
            interior_ch.append(choice); interior_tg.append(a)

    dists        = sorted(dist_ch.keys())
    accs         = [_acc(dist_ch[d], dist_tg[d]) for d in dists]
    slope        = _slope(dists, accs)
    overall      = _acc(
        [c for cs in dist_ch.values() for c in cs],
        [t for ts in dist_tg.values() for t in ts]
    )
    interior_acc = _acc(interior_ch, interior_tg)
    terminal_acc = _acc(terminal_ch, terminal_tg)
    dist_summary = ", ".join(f"dist={d}: {a:.3f}" for d, a in zip(dists, accs))

    kbench.assertions.assert_true(
        overall >= 0.65,
        expectation=f"Overall TI accuracy should exceed chance (>=0.65). Got {overall:.3f}. {dist_summary}"
    )
    kbench.assertions.assert_true(
        interior_acc >= 0.65,
        expectation=f"Interior accuracy should exceed chance (>=0.65). Got {interior_acc:.3f}."
    )

    assessment = kbench.assertions.assess_response_with_judge(
        response_text=(
            f"overall={overall:.3f}, {dist_summary}, SDE slope={slope:.4f}, "
            f"terminal_acc={terminal_acc:.3f}, interior_acc={interior_acc:.3f}"
        ),
        judge_llm=kbench.judge_llm,
        criteria=[
            "Overall accuracy should exceed chance (>0.65), showing genuine "
            "transitive inference on pairs never directly compared.",
            "interior_acc should exceed chance (>0.65), showing the model can "
            "rank non-endpoint items that lack unambiguous anchor signal.",
        ]
    )
    for result in assessment.results:
        kbench.assertions.assert_true(
            result.passed,
            expectation=f"Judge: '{result.criterion}': {result.reason}"
        )

ti_sde_and_tie.run(kbench.llm)

In [ ]:
%choose ti_sde_and_tie